# CMSC 173 &middot; Module 8 &middot; Example 5
## A forest of earthquake trees

Example 4's unlimited tree memorised the training quakes. Here 200 such trees, each grown on its own bootstrap sample, are averaged; then boosting adds small trees one at a time. Part 10's two by-hand examples (a bootstrap of the nine tennis days, boosting on the five condos) come first.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

**A bootstrap sample of the nine tennis days.** Nine draws from days 1 to 9, *with replacement*.

In [ ]:
import numpy as np

draw = np.random.default_rng(173).integers(1, 10, size=9)
print(sorted(draw.tolist()))                                      # [1, 3, 4, 5, 5, 6, 7, 8, 9]
print("never drawn:", sorted(set(range(1, 10)) - set(draw.tolist())))   # [2]
for n in (9, 714, 7973):
    print(n, "rows: share left out of a sample", round((1 - 1 / n) ** n, 3))   # 0.346, 0.368, 0.368

**Boosting by hand on module 3's five condos.** Start from the mean; each round, fit a stump to the residuals and add half of it (learning rate 0.5).

In [ ]:
from sklearn.tree import DecisionTreeRegressor

area = np.array([[20], [40], [60], [80], [100]])      # sqm
price = np.array([2, 4, 5, 4, 5.0])                   # PHP millions
pred = np.full(5, price.mean())                       # round 0: 4.00 for every condo
print(0, pred.round(2), "SSE", round(((price - pred) ** 2).sum(), 2))
for r in (1, 2, 3):
    stump = DecisionTreeRegressor(max_depth=1).fit(area, price - pred)   # fit what is still wrong
    pred = pred + 0.5 * stump.predict(area)                              # keep half of the correction
    print(r, "area <=", stump.tree_.threshold[0], "| leaves", stump.tree_.value[1:, 0, 0].round(3),
          "|", pred.round(2), "SSE", round(((price - pred) ** 2).sum(), 2))
# SSE 6.0 -> 2.25 -> 1.27 -> 0.83

**Example 4's quakes, Example 4's split.**

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/njpinton/cmsc173-labs/main/data/ph_earthquakes.csv"
quakes = pd.read_csv(url, parse_dates=["time"])
X, y = quakes[["longitude", "latitude"]], quakes["depth"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=173)
print(len(X_tr), "training quakes,", len(X_te), "test quakes")

**One deep tree is unstable.** Grow two unlimited trees on two bootstrap samples and compare what they predict for the same test quakes.

In [ ]:
rng = np.random.default_rng(173)
preds = []
for _ in range(2):
    rows = rng.integers(0, len(X_tr), len(X_tr))       # a bootstrap sample of row positions
    tree = DecisionTreeRegressor(random_state=173).fit(X_tr.iloc[rows], y_tr.iloc[rows])
    preds.append(tree.predict(X_te))
print("two bootstrap trees disagree by", round(np.abs(preds[0] - preds[1]).mean()), "km on average")   # 28

**One tree, a forest, and boosting.** With two features, the forest's regression default lets every split use both, so this forest is plain bagging.

In [ ]:
models = {"one tree, no depth limit": DecisionTreeRegressor(random_state=173),
          "one tree, depth 6": DecisionTreeRegressor(max_depth=6, random_state=173),
          "random forest, 200 trees": RandomForestRegressor(n_estimators=200, random_state=173, n_jobs=-1),
          "random forest, min_samples_leaf=5": RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                                     random_state=173, n_jobs=-1),
          "gradient boosting": GradientBoostingRegressor(random_state=173)}
for name, m in models.items():
    print(f"{name:35s} test R2 {m.fit(X_tr, y_tr).score(X_te, y_te):.3f}")
# 0.401 | 0.621 | 0.625 | 0.656 | 0.601

**More trees help, up to a point.**

In [ ]:
import matplotlib.pyplot as plt

counts = [1, 5, 10, 25, 50, 100, 200, 400]
scores = [RandomForestRegressor(n_estimators=n, random_state=173, n_jobs=-1).fit(X_tr, y_tr).score(X_te, y_te)
          for n in counts]
print(" ".join(f"{s:.3f}" for s in scores))     # 0.419 0.611 0.614 0.621 0.625 0.621 0.625 0.626
# (scikit-learn 1.4.2; newer versions can differ in the third decimal, e.g. 0.610 for five trees)
plt.plot(counts, scores, marker="o", label="random forest")
plt.axhline(0.621, ls="--", color="orange", label="one tree, depth 6")
plt.axhline(0.401, ls=":", color="red", label="one tree, no limit")
plt.xscale("log"); plt.xlabel("trees in the forest"); plt.ylabel("test R2"); plt.legend()
plt.show()

**When ensembles win.** Example 3's Titanic (three features) and scikit-learn's handwritten digits (64).

In [ ]:
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age"]].dropna()
df["female"] = (df["sex"] == "female").astype(int)
Xt, yt = df[["female", "pclass", "age"]], df["survived"]
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(Xt, yt, test_size=0.3, random_state=173, stratify=yt)
cv = StratifiedKFold(5, shuffle=True, random_state=173)
for name, m in {"tree, depth 3": DecisionTreeClassifier(max_depth=3, random_state=173),
                "forest, 200 trees, depth 5": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=173),
                "gradient boosting": GradientBoostingClassifier(random_state=173)}.items():
    print(f"Titanic  {name:27s} test {m.fit(Xt_tr, yt_tr).score(Xt_te, yt_te):.3f}"
          f"   5-fold {cross_val_score(m, Xt, yt, cv=cv).mean():.3f}")
# test 0.795 / 0.781 / 0.763; 5-fold 0.791 / 0.803 / 0.811

Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=173, stratify=yd)
for name, m in {"tree, no depth limit": DecisionTreeClassifier(random_state=173),
                "forest, 200 trees": RandomForestClassifier(n_estimators=200, random_state=173, n_jobs=-1),
                "gradient boosting": GradientBoostingClassifier(random_state=173)}.items():
    print(f"digits   {name:27s} test {m.fit(Xd_tr, yd_tr).score(Xd_te, yd_te):.3f}")
# 0.830 / 0.974 / 0.969

**Try it.** Give the quake forest `max_features=1`: each split then sees only one of the two features, chosen at random, which makes it a true random forest rather than bagging. Does the test R2 go up or down?